## NASA Earthdata Login credential is needed to use earthaccess package
Create `.netrc` with info below under home directory

`machine urs.earthdata.nasa.gov login [account] password [password]`

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import earthaccess
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mpcrs
import cartopy.crs as ccrs
import cartopy.feature as cft
from matplotlib.colors import LinearSegmentedColormap

In [2]:
def modisaod_to_dataset(modisgranule):
    import xarray as xr
    from pyhdf.SD import SD, SDC
    hdf = SD(modisgranule, SDC.READ)

    # All of MODIS AOD data have a singular reference time - good practice to get from attribute
    modis_time_key = 'Scan_Start_Time'
    try:
        modis_time_attribute = hdf.select(modis_time_key).attributes().get('units')
        if modis_time_attribute is None:
            print("'units' attribute is not present in {modis_time_key}.")
            modis_ref_time = datetime(1993, 1, 1, 0, 0, 0)
        else:
            # Extract the date and time part
            datetime_str = modis_time_attribute.split('since ')[1].rsplit(' ', 1)[0]

            # Convert to a datetime object
            modis_ref_time = datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S.%f")
    except Exception as e:
        # Catch and print any errors
        print(f"An error occurred: {e}")
    #  Get variables
    modis_time = hdf.select(modis_time_key)[:].ravel()
    cnts = len(modis_time)

    land_sea_flag = hdf.select('Land_sea_Flag')[:].ravel()
    aod = hdf.select('AOD_550_Dark_Target_Deep_Blue_Combined')[:].ravel() * 1e-3
    unc_land = hdf.select('Deep_Blue_Aerosol_Optical_Depth_550_Land_Estimated_Uncertainty')[:].ravel() * 1e-3
    over_land = np.logical_not(land_sea_flag == 0)
    
    data_dict = {
        'lat': (['Location'], hdf.select('Latitude')[:].ravel()),
        'lon': (['Location'], hdf.select('Longitude')[:].ravel()),
        'aod': (['Location'], aod),
        'land_sea_flag': (['Location'], land_sea_flag),
        'QC_flag': (['Location'], hdf.select('Land_Ocean_Quality_Flag')[:].ravel()),
        'sol_zen': (['Location'], hdf.select('Solar_Zenith')[:].ravel()),
        'sen_zen': (['Location'], hdf.select('Sensor_Zenith')[:].ravel()),
        'uncertainty': (['Location'], np.where(over_land, unc_land, np.add(0.05, np.multiply(0.15, aod)))),
        'obs_time': (['Location'], (modis_time + modis_ref_time.timestamp()).astype('datetime64[s]')),
    }

    coords_dict = {'Location': np.arange(cnts)}
    return xr.Dataset(data_dict, coords=coords_dict)

In [3]:
def merra2_gridcell_area(lat):
    """
    Compute MERRA-2 grid-cell area for a 0.5° x 0.625° lat-lon grid.

    Parameters
    ----------
    lat : array-like
        Latitude centers in degrees (-90 to 90, step 0.5)

    Returns
    -------
    area : ndarray
        Grid-cell area in m^2 with shape (lat,)
        (same for all longitudes at a given latitude)
    """
    R = 6_371_000.0  # Earth radius [m]

    dlat = np.deg2rad(0.5)
    dlon = np.deg2rad(0.625)

    lat_rad = np.deg2rad(lat)

    area = (
        R**2
        * dlon
        * (np.sin(lat_rad + dlat / 2) - np.sin(lat_rad - dlat / 2))
    )

    return area

In [4]:
tags_dict = {
    'extinction': 'M2T1NXAER',
    'emission': 'M2T1NXADG',
}
projection = ccrs.PlateCarree()

In [ ]:
# Get extinction files
results = earthaccess.search_data(short_name=tags_dict['extinction'], temporal=('2024-10-15', '2024-11-30'))
files = earthaccess.open(results)

In [ ]:
ds = xr.open_dataset(files[0])
ds

In [ ]:
# Plot total AOD
fig, ax = plt.subplots(subplot_kw=dict(projection=projection))
ds['TOTEXTTAU'].isel(time=0).plot.contourf(
    ax=ax,
    levels=np.arange(0, 1.6, 0.1),
    cmap='ocean_r',
    cbar_kwargs={'fraction':0.025}
)
ax.coastlines(color='grey')

In [5]:
# Get emission and other diagnose files
emissresults = earthaccess.search_data(short_name=tags_dict['emission'], temporal=('2024-10-15', '2024-11-30'))
emissfiles = earthaccess.open(emissresults)

QUEUEING TASKS | :   0%|          | 0/47 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/47 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/47 [00:00<?, ?it/s]

In [6]:
def global_emission(ds, varlist):
    m2area = merra2_gridcell_area(ds.lat.values)
    area_2d = m2area[:, None]
    df = pd.DataFrame({'time': ds.time.values})
    for var in varlist:
        df[var] = (ds[var] * area_2d).sum(dim=('lat', 'lon')).values
    return df
    
def global_average(ds, varlist):
    df = pd.DataFrame({'time': ds.time.values})
    for var in varlist:
        df[var] = ds[var].mean(dim=('lat', 'lon')).data
    return df

In [11]:
all_ds = xr.open_mfdataset(emissfiles, engine="h5netcdf", combine="by_coords")
all_ds

KeyboardInterrupt: 

In [ ]:
all_ds['BCEMBB'].mean(dim=('lat', 'lon')).values

In [ ]:
ts_df = pd.DataFrame()
ts_df = global_average(all_ds, ['BCEMBB', 'OCEMBB', 'SUEXTTAU'])
ts_df = ts_df.set_index('time')

In [ ]:
fig, ax = plt.subplots(
    # subplot_kw={'figsize': (10, 5)}
)

cols = ts_df.columns
ax = ts_df['SUEXTTAU'].plot(ax=ax)    
# ax2 = ts_df['BCEMBB'].plot(ax=ax, secondary_y=True)

fig.legend()


In [ ]:
fig, ax = plt.subplots(subplot_kw=dict(projection=projection))
tmpds['SUEXTTAU'].isel(time=23).plot.contourf(
    ax=ax,
    levels=np.arange(0, 1.6, 0.1),
    cmap='ocean_r',
    cbar_kwargs={'fraction':0.025}
)
ax.coastlines(color='grey')

In [ ]:
tmpdf['EMBB'] = ts_df['BCEMBB'] + ts_df['OCEMBB']

In [ ]:
totaldf = pd.DataFrame()
totaldf['EMBB'] = ts_df['BCEMBB'] + ts_df['OCEMBB']

In [ ]:
fig, ax = plt.subplots()
ax = (totaldf['EMBB']).plot(figsize=(10, 4))
fig.legend()